# Lab 8 — AI-Based Biometric Identification System
### Face · Iris · Fingerprint recognition

**Course:** Artificial Intelligence in Cybersecurity  **Environment:** Google Colab / Python

| Assignment | What it does | Data |
|---|---|---|
| 1 | Build & verify our own face dataset | `data/student_faces/` (team photos) |
| 2 | Interactive face recognition app (Gradio) | dataset from Assignment 1 |
| 3 | Iris segmentation + IrisCode / ORB matching | CASIA-IrisV1 subset |
| 4 | Fingerprint enhancement + ORB/RANSAC matching | SOCOFing subset |
| 5 | Comparative analysis of the three modalities | results of 2–4 |

**How to run:** `Runtime → Run all`. If a dataset folder is missing, the notebook asks you to upload
its ZIP (`student_faces.zip`, `iris_dataset.zip`, `fingerprint_dataset.zip`). Every result
(CSV files, figures) is written to the project root and to `results/`.

> **Team:** fill in the names and roles in `report/final_report.md` (section 3).

## 0. Setup
Three key ideas used everywhere below:
* **Detection** — *where* is the face / iris / fingerprint in the image?
* **Verification (1:1)** — are these **two** samples the same person? → MATCH / NO MATCH
* **Identification (1:N)** — *who* is this, searching the whole database? → name or UNKNOWN

In [ ]:
# --- 0.1 Install libraries (only needed on Google Colab; locally use requirements.txt) ---
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # deepface = face detection + pretrained embedding models; gradio = web UI
    !pip -q install deepface gradio

# Hide TensorFlow's very noisy start-up logs (they are not errors)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
print("Running on Google Colab:", IN_COLAB)

In [ ]:
# --- 0.2 Imports ---
import csv, glob, itertools, random, shutil, warnings, zipfile
from datetime import datetime
from pathlib import Path

import cv2                      # OpenCV: image processing (blur, threshold, ORB, Hough...)
import numpy as np              # vectors / matrices
import pandas as pd             # result tables -> CSV
import matplotlib.pyplot as plt # all visualisations
from PIL import Image, ImageOps # image loading (+ fixes phone EXIF rotation)
from skimage.morphology import skeletonize  # thin ridges to 1-pixel lines (fingerprints)

warnings.filterwarnings("ignore")
try:
    display                     # available automatically inside Jupyter / Colab
except NameError:
    display = print             # fallback when the code is run as a plain script
random.seed(42); np.random.seed(42)   # reproducible experiments
plt.rcParams["figure.dpi"] = 80        # smaller inline figures (saved PNGs use dpi=120)

In [ ]:
# --- 0.3 Configuration: ALL tunable parameters live here (change them, not the code) ---
PROJECT_ROOT = Path.cwd()                     # Colab: /content, locally: the repo folder
DATA_DIR     = PROJECT_ROOT / "data"
RESULTS_DIR  = PROJECT_ROOT / "results"

FACES_DIR  = DATA_DIR / "student_faces"       # Assignment 1-2 (our own photos)
GROUP_DIR  = DATA_DIR / "group_photos"        # optional: group photos for batch evaluation
IRIS_DIR   = DATA_DIR / "iris_dataset"        # Assignment 3
FINGER_DIR = DATA_DIR / "fingerprint_dataset" # Assignment 4

# Face recognition
MODEL_NAME     = "Facenet512"   # pretrained CNN -> 512-number face embedding
DETECTOR       = "opencv"       # face detector backend ("retinaface"/"mtcnn" are more accurate but slower)
FACE_THRESHOLD = 0.30           # cosine distance above this -> UNKNOWN (justified in section 2.5)
MIN_FACE_SIZE  = 40             # px; smaller detections are treated as false positives
REGISTRATION_FILES = ["photo_01.jpg", "photo_02.jpg", "photo_03.jpg", "photo_04.jpg"]
TEST_FILE          = "test_05.jpg"   # NEVER used for registration

# Iris recognition
IRIS_HD_THRESHOLD  = 0.32       # IrisCode Hamming distance below this -> MATCH (section 3.8)
IRIS_ORB_THRESHOLD = 15         # ORB good matches >= this -> MATCH (lab's demo value)

# Fingerprint recognition
FP_UPSCALE          = 3         # SOCOFing images are ~96x103 px -> enlarge before ORB
FP_ORB_THRESHOLD    = 20        # ORB good matches >= this -> MATCH (lab's demo value)
FP_INLIER_THRESHOLD = 12        # RANSAC-verified matches >= this -> MATCH (our improvement)

for sub in ["face", "iris/crops", "fingerprint", "comparison"]:
    (RESULTS_DIR / sub).mkdir(parents=True, exist_ok=True)

def save_fig(name: str, sub: str):
    """Save the current matplotlib figure to results/<sub>/<name>.png and show it."""
    path = RESULTS_DIR / sub / f"{name}.png"
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()
    print("saved ->", path.relative_to(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
# --- 0.4 Dataset loader: use the folder if it exists, otherwise ask for a ZIP upload (Colab) ---
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

def folder_has_images(folder: Path) -> bool:
    return folder.is_dir() and any(p.suffix.lower() in IMG_EXT for p in folder.rglob("*"))

def safe_extract(zip_path: str, dest: Path, expected_root: str):
    """Extract a ZIP safely.
    * blocks 'zip-slip' (paths like ../../etc/passwd) — a classic AppSec bug
    * skips macOS junk (__MACOSX, .DS_Store)
    * works whether the ZIP contains 'student_faces/...' or just the sub-folders
    """
    with zipfile.ZipFile(zip_path) as archive:
        names = [n for n in archive.namelist() if "__MACOSX" not in n and not n.endswith(".DS_Store")]
        has_root = all(n.startswith(expected_root + "/") for n in names if n.strip("/"))
        target = dest if has_root else dest / expected_root
        for name in names:
            out = (target / name).resolve()
            if not str(out).startswith(str(target.resolve())):
                raise ValueError(f"Unsafe path inside ZIP: {name}")
            archive.extract(name, target)

def ensure_dataset(folder: Path) -> Path:
    """Return `folder` if it contains images, otherwise upload <folder>.zip (Colab)."""
    if folder_has_images(folder):
        return folder
    if IN_COLAB:
        from google.colab import files
        print(f"'{folder.name}' not found -> please select {folder.name}.zip")
        uploaded = files.upload()                      # opens the file picker
        zip_name = next(iter(uploaded))
        safe_extract(zip_name, folder.parent, folder.name)
    if not folder_has_images(folder):
        raise FileNotFoundError(f"No images in {folder}. See README -> Setup guide.")
    print("Dataset uploaded successfully!")
    return folder

def load_rgb(path) -> np.ndarray:
    """Load any image as an RGB uint8 array. exif_transpose() rotates phone photos correctly."""
    return np.array(ImageOps.exif_transpose(Image.open(path)).convert("RGB"))

---
# ASSIGNMENT 1 — Custom facial image dataset
Each student folder holds 5 photos: `photo_01..04.jpg` (registration: frontal, right turn,
left turn, other lighting) and `test_05.jpg` (testing only).

*Analogy:* registration photos are like giving a guard four ID pictures of you from different
angles; the test photo is you walking up to the door later — the guard must recognise you from
a picture he has **never** seen.

In [ ]:
# --- 1.1 Load (or upload) the dataset ---
dataset_path = ensure_dataset(FACES_DIR)
print("Dataset folder found:", dataset_path.is_dir())

In [ ]:
# --- 1.2 Verify the structure: number of students and photos (Code Example 2) ---
students = sorted(p.name for p in dataset_path.iterdir() if p.is_dir())
print("Number of students:", len(students), "\n")
for student in students:
    images = [f for f in os.listdir(dataset_path / student) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    print(f"{student} — {len(images)} photographs")

# Show the folder tree (useful for the report / defense)
print("\nstudent_faces/")
for student in students:
    print(f"├── {student}/")
    for f in sorted(os.listdir(dataset_path / student)):
        print(f"│     ├── {f}")

In [ ]:
# --- 1.3 Quality check against the lab's technical requirements ---
# Requirements: JPG/JPEG, >= 640x480, RGB, exactly ONE face, file names as specified.
from deepface import DeepFace   # imported here because Assignment 1 already uses its detector

quality_rows = []
for student in students:
    for fname in REGISTRATION_FILES + [TEST_FILE]:
        path = dataset_path / student / fname
        row = {"student": student, "file": fname, "exists": path.exists()}
        if path.exists():
            with Image.open(path) as im:
                w, h = ImageOps.exif_transpose(im).size
                row.update(format=im.format, mode=im.mode, width=w, height=h)
            try:   # count faces with the same detector the recogniser uses
                faces = DeepFace.extract_faces(img_path=str(path), detector_backend=DETECTOR,
                                               enforce_detection=True)
                row["faces_found"] = len(faces)
            except ValueError:
                row["faces_found"] = 0
            row["resolution_ok"] = max(w, h) >= 640 and min(w, h) >= 480
            row["valid"] = (row["format"] == "JPEG" and row["mode"] == "RGB"
                            and row["resolution_ok"] and row["faces_found"] == 1)
        quality_rows.append(row)

quality = pd.DataFrame(quality_rows)
quality.to_csv(RESULTS_DIR / "face" / "dataset_quality.csv", index=False)
print("Valid photos:", int(quality["valid"].sum()), "/", len(quality))
quality

In [ ]:
# --- 1.4 Visualise every student's 5 photos (Code Example 3, extended to all students) ---
for student in students:
    plt.figure(figsize=(15, 3.6))
    for i, fname in enumerate(REGISTRATION_FILES + [TEST_FILE]):
        plt.subplot(1, 5, i + 1)
        plt.imshow(load_rgb(dataset_path / student / fname))
        plt.title(fname + ("  (TEST)" if fname == TEST_FILE else ""))
        plt.axis("off")
    plt.suptitle("Student photographs: " + student)
    plt.tight_layout()
    save_fig(f"dataset_{student}", "face")

---
# ASSIGNMENT 2 — Interactive face recognition system
**Pipeline** (same for registration and recognition — consistency matters!):
```
photo ──► detect faces ──► crop + align each face ──► FaceNet512 ──► 512-D embedding
                                                                        │
           registered database {name: [4 embeddings]} ◄── compare (cosine distance)
                                                                        │
                                    distance ≤ threshold ? name : "UNKNOWN"
```
An **embedding** is a "numeric fingerprint of a face": 512 numbers such that photos of the same
person land close together and different people land far apart.

In [ ]:
# --- 2.1 Load the pretrained model once (weights ~95 MB are downloaded on first run) ---
DeepFace.build_model(MODEL_NAME)
print("Model ready:", MODEL_NAME, "| detector:", DETECTOR)

In [ ]:
# --- 2.2 One function that turns an RGB image into (face boxes, embeddings) ---
def detect_and_embed(image_rgb: np.ndarray):
    """Detect ALL faces in an RGB image and return a list of dicts:
       {"box": (x, y, w, h), "embedding": np.ndarray(512), "confidence": float}
    """
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)   # DeepFace/OpenCV expect BGR arrays
    try:
        faces = DeepFace.extract_faces(img_path=image_bgr, detector_backend=DETECTOR,
                                       enforce_detection=True, align=True)
    except ValueError:            # raised when no face is found
        return []
    results = []
    for face in faces:
        area = face["facial_area"]
        if min(area["w"], area["h"]) < MIN_FACE_SIZE:   # tiny box -> probably not a face
            continue
        # extract_faces returns an RGB float image in [0,1]; convert to uint8 BGR for represent()
        # (the lab's example passes RGB directly -> red/blue swapped -> worse embeddings)
        face_bgr = cv2.cvtColor((np.clip(face["face"], 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
        embedding = DeepFace.represent(img_path=face_bgr, model_name=MODEL_NAME,
                                       detector_backend="skip", enforce_detection=False)[0]["embedding"]
        results.append({"box": (area["x"], area["y"], area["w"], area["h"]),
                        "embedding": np.asarray(embedding, dtype=np.float32),
                        "confidence": float(face.get("confidence") or 0)})
    return results

In [ ]:
# --- 2.3 Registration: build the database from photo_01..04 ONLY (Code Example 5) ---
database = {}   # {student_name: [embedding, embedding, ...]}

for student in students:
    embeddings = []
    for filename in REGISTRATION_FILES:
        assert filename != TEST_FILE          # safety net: the test photo must never be registered
        image_path = dataset_path / student / filename
        if not image_path.exists():
            print("Missing:", image_path); continue
        faces = detect_and_embed(load_rgb(image_path))
        if len(faces) == 0:
            print("Skipped (no face):", image_path); continue
        if len(faces) > 1:   # should not happen for a single-person photo -> keep the biggest face
            print(f"Warning: {len(faces)} faces in {image_path.name}, using the largest")
        largest = max(faces, key=lambda f: f["box"][2] * f["box"][3])
        embeddings.append(largest["embedding"])
    if embeddings:
        database[student] = embeddings

print("Registered students:", len(database), "\n")
for name, vectors in database.items():
    print(f"{name} — {len(vectors)} samples")

In [ ]:
# --- 2.4 Matching: cosine distance + nearest-neighbour search (Code Example 6) ---
def cosine_distance(a, b) -> float:
    """0 = identical direction, 1 = unrelated, 2 = opposite.  distance = 1 - cos(angle)."""
    a = np.asarray(a, dtype=np.float32); b = np.asarray(b, dtype=np.float32)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 1.0
    return 1.0 - float(np.dot(a, b) / denominator)

def recognize_embedding(query_vector, database, threshold=None):
    """1:N identification. Return (best_name or 'UNKNOWN', best_distance)."""
    threshold = FACE_THRESHOLD if threshold is None else threshold
    best_name, best_distance = "UNKNOWN", float("inf")
    for student, vectors in database.items():
        for registered_vector in vectors:
            distance = cosine_distance(query_vector, registered_vector)
            if distance < best_distance:
                best_distance, best_name = distance, student
    if best_distance > threshold:          # closest match is still "too far" -> reject
        best_name = "UNKNOWN"
    return best_name, best_distance

### 2.5 Choosing the threshold experimentally
We need to know how far apart *same-person* (genuine) and *different-person* (impostor)
embeddings are **for our data and our model**:
* **Genuine distances:** each `test_05.jpg` vs. its own 4 registered embeddings.
* **Impostor distances:** each `test_05.jpg` vs. every *other* student's embeddings —
  this simulates an unregistered person walking in.

For every candidate threshold we measure **FRR** (False Rejection Rate — a registered student
is called UNKNOWN) and **FAR** (False Acceptance Rate — a stranger is accepted as someone).
A security system prefers a low FAR; a convenience system prefers a low FRR.

In [ ]:
genuine, impostor, test_rows = [], [], []
for student in database:
    test_path = dataset_path / student / TEST_FILE
    faces = detect_and_embed(load_rgb(test_path)) if test_path.exists() else []
    if not faces:
        print("No face in", test_path); continue
    query = max(faces, key=lambda f: f["box"][2] * f["box"][3])["embedding"]
    for other, vectors in database.items():
        d = min(cosine_distance(query, v) for v in vectors)   # distance to the closest sample
        (genuine if other == student else impostor).append(d)
    name, dist = recognize_embedding(query, database)
    test_rows.append({"student": student, "predicted": name, "distance": round(dist, 4),
                      "correct": name == student})

test_df = pd.DataFrame(test_rows)
print("Identification accuracy on held-out test_05 photos:",
      f"{test_df['correct'].mean() * 100:.1f}%")
display(test_df)

thresholds = np.round(np.arange(0.05, 0.81, 0.01), 2)
far = [np.mean(np.array(impostor) <= t) * 100 for t in thresholds]
frr = [np.mean(np.array(genuine) > t) * 100 for t in thresholds]
sweep = pd.DataFrame({"threshold": thresholds, "FAR_%": far, "FRR_%": frr})
sweep.to_csv(RESULTS_DIR / "face" / "threshold_sweep.csv", index=False)

# a threshold is "safe" if it lies between the worst genuine and the best impostor distance
lo, hi = max(genuine), min(impostor)
print(f"max genuine distance = {lo:.3f} | min impostor distance = {hi:.3f}")
if lo < hi:
    print(f"Every threshold in ({lo:.2f}, {hi:.2f}) separates our data perfectly; "
          f"current FACE_THRESHOLD = {FACE_THRESHOLD}")
else:
    best = sweep.iloc[(sweep['FAR_%'] - sweep['FRR_%']).abs().argmin()]
    print(f"Classes overlap. Equal-error threshold ≈ {best.threshold} -> consider updating FACE_THRESHOLD")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(genuine, bins=15, alpha=.7, label="genuine (same person)", color="tab:green")
ax[0].hist(impostor, bins=15, alpha=.7, label="impostor (other person)", color="tab:red")
ax[0].axvline(FACE_THRESHOLD, ls="--", c="k", label=f"threshold {FACE_THRESHOLD}")
ax[0].set_xlabel("cosine distance"); ax[0].set_title("Distance distributions"); ax[0].legend()
ax[1].plot(thresholds, far, label="FAR %", c="tab:red"); ax[1].plot(thresholds, frr, label="FRR %", c="tab:green")
ax[1].axvline(FACE_THRESHOLD, ls="--", c="k"); ax[1].set_xlabel("threshold"); ax[1].set_ylabel("%")
ax[1].set_title("Error rates vs threshold"); ax[1].legend()
save_fig("threshold_analysis", "face")

### 2.6 Recognising a group photograph (Code Example 7 — improved)
Improvements over the lab's example: BGR/RGB fix, tiny-box filter, distance printed next to
each name, scalable line width/font for large phone photos, and a per-face report.

In [ ]:
def recognize_uploaded_photo(photo):
    """Gradio callback: PIL image in -> (annotated PIL image, text summary) out."""
    if photo is None:
        return None, "Please upload a photograph."
    image = np.array(ImageOps.exif_transpose(photo).convert("RGB"))
    faces = detect_and_embed(image)
    if not faces:
        return photo, "No faces detected in the photograph."

    output = image.copy()
    scale = max(image.shape[:2]) / 1000             # thicker boxes on big photos
    thick, font = max(2, int(3 * scale)), max(0.6, 0.8 * scale)
    recognized, unknown, lines = 0, 0, []
    # sort faces left -> right so the text report reads like the photo
    for i, face in enumerate(sorted(faces, key=lambda f: f["box"][0]), start=1):
        x, y, w, h = face["box"]
        name, distance = recognize_embedding(face["embedding"], database)
        if name == "UNKNOWN":
            unknown += 1; color = (255, 0, 0)          # red (RGB)
        else:
            recognized += 1; color = (0, 200, 0)       # green
        cv2.rectangle(output, (x, y), (x + w, y + h), color, thick)
        label = f"{name} ({distance:.2f})"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font, thick)
        ty = max(y - 10, th + 10)
        cv2.rectangle(output, (x, ty - th - 8), (x + tw + 6, ty + 4), color, -1)  # label background
        cv2.putText(output, label, (x + 3, ty), cv2.FONT_HERSHEY_SIMPLEX, font, (255, 255, 255), thick)
        lines.append(f"  face {i}: {name:<10} distance={distance:.3f}")

    summary = (f"Detected faces: {len(faces)}\nRecognized students: {recognized}\n"
               f"Unknown faces: {unknown}\n\nPer face (left → right):\n" + "\n".join(lines))
    return Image.fromarray(output), summary

### 2.7 Batch evaluation of group photos → `face_results.csv`
If `data/group_photos/` contains photos **and** a `ground_truth.csv` (columns `photo,
actual_people, registered_names, unregistered_people`) we evaluate them automatically.
During the live defense the Gradio app (2.8) appends rows to the same CSV.

Counting rule (no face-to-name position matching needed):
* **correct** = predicted names that really are in the photo (each name counted once)
* **incorrect** = predicted names of students who are *not* in the photo, or duplicates
* **Accuracy** = correct / registered students actually present × 100 %

In [ ]:
FACE_RESULTS_CSV = PROJECT_ROOT / "face_results.csv"
FACE_COLUMNS = ["photo", "actual_people", "registered_present", "detected_faces", "correct",
                "incorrect", "unknown", "missed_registered", "accuracy_%", "predicted_names", "timestamp"]

def evaluate_prediction(photo_name, predicted, actual_people, registered_present):
    """Compare predicted names with the ground-truth list -> one result row."""
    remaining = list(registered_present)
    correct = incorrect = unknown = 0
    for name in predicted:
        if name == "UNKNOWN":
            unknown += 1
        elif name in remaining:
            correct += 1; remaining.remove(name)
        else:
            incorrect += 1
    n_reg = len(registered_present)
    return {"photo": photo_name, "actual_people": actual_people, "registered_present": n_reg,
            "detected_faces": len(predicted), "correct": correct, "incorrect": incorrect,
            "unknown": unknown, "missed_registered": n_reg - correct,
            "accuracy_%": round(100 * correct / n_reg, 1) if n_reg else None,
            "predicted_names": ";".join(predicted), "timestamp": datetime.now().isoformat(timespec="seconds")}

def append_face_result(row):
    new = not FACE_RESULTS_CSV.exists()
    with open(FACE_RESULTS_CSV, "a", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=FACE_COLUMNS)
        if new: writer.writeheader()
        writer.writerow(row)

gt_file = GROUP_DIR / "ground_truth.csv"
if gt_file.exists():
    if FACE_RESULTS_CSV.exists():
        FACE_RESULTS_CSV.unlink()                      # fresh batch run
    gt = pd.read_csv(gt_file)
    for _, g in gt.iterrows():
        photo = Image.open(GROUP_DIR / g["photo"])
        annotated, summary = recognize_uploaded_photo(photo)
        annotated.save(RESULTS_DIR / "face" / f"{Path(g['photo']).stem}_result.jpg")
        image = np.array(ImageOps.exif_transpose(photo).convert("RGB"))
        predicted = [recognize_embedding(f["embedding"], database)[0]
                     for f in sorted(detect_and_embed(image), key=lambda f: f["box"][0])]
        names = [n for n in str(g["registered_names"]).split(";") if n and n != "nan"]
        append_face_result(evaluate_prediction(g["photo"], predicted, int(g["actual_people"]), names))

    face_results = pd.read_csv(FACE_RESULTS_CSV)
    total_correct, total_registered = face_results["correct"].sum(), face_results["registered_present"].sum()
    FACE_ACCURACY = 100 * total_correct / total_registered
    print(f"Overall identification accuracy: {total_correct}/{total_registered} = {FACE_ACCURACY:.1f}%")
    display(face_results)

    # show all annotated results in one figure
    n = len(gt)
    plt.figure(figsize=(16, 3.2 * ((n + 1) // 2)))
    for i, name in enumerate(gt["photo"]):
        plt.subplot((n + 1) // 2, 2, i + 1)
        plt.imshow(Image.open(RESULTS_DIR / "face" / f"{Path(name).stem}_result.jpg"))
        plt.title(name); plt.axis("off")
    plt.tight_layout()
    save_fig("group_results_overview", "face")
else:
    FACE_ACCURACY = None
    print("No data/group_photos/ground_truth.csv -> use the Gradio app below; results are appended to face_results.csv")

### 2.8 Interactive application (Gradio) — used during the live defense
Upload any new group photo → **RECOGNIZE FACES**. No code changes or restarts are needed
between photos. The optional *Evaluation* panel logs the result into `face_results.csv`.

In [ ]:
import gradio as gr

def log_result(photo, actual_people, present_names):
    """Gradio callback for the evaluation panel: recompute predictions and append a CSV row."""
    if photo is None:
        return "Upload and recognise a photo first."
    image = np.array(ImageOps.exif_transpose(photo).convert("RGB"))
    predicted = [recognize_embedding(f["embedding"], database)[0]
                 for f in sorted(detect_and_embed(image), key=lambda f: f["box"][0])]
    names = [n.strip() for n in (present_names or "").split(",") if n.strip()]
    unknown_names = [n for n in names if n not in database]
    if unknown_names:
        return f"Not registered: {unknown_names}. Registered: {list(database)}"
    photo_id = f"live_{datetime.now().strftime('%H%M%S')}.jpg"
    row = evaluate_prediction(photo_id, predicted, int(actual_people or len(predicted)), names)
    append_face_result(row)
    return (f"Saved to face_results.csv as {photo_id}: correct {row['correct']}/{row['registered_present']}, "
            f"incorrect {row['incorrect']}, unknown {row['unknown']}, accuracy {row['accuracy_%']}%")

with gr.Blocks(title="Face Recognition System") as demo:
    gr.Markdown("# FACE RECOGNITION SYSTEM")
    gr.Markdown("Upload a group photograph to identify registered students.")
    gr.Markdown(f"**Registered students: {len(database)}** — {', '.join(database)}  \n"
                f"Model: {MODEL_NAME} · Detector: {DETECTOR} · Threshold: {FACE_THRESHOLD}")
    with gr.Row():
        input_photo = gr.Image(label="Upload Photo", type="pil", sources=["upload"])
        output_photo = gr.Image(label="Recognition Result")
    recognize_button = gr.Button("RECOGNIZE FACES", variant="primary")
    result_text = gr.Textbox(label="Identification Results", lines=9)
    with gr.Accordion("Evaluation (optional) — save this photo's result to face_results.csv", open=False):
        actual_people = gr.Number(label="Actual number of people in the photo", precision=0)
        present_names = gr.Textbox(label="Registered students actually present (comma-separated)",
                                   placeholder=", ".join(database))
        save_button = gr.Button("SAVE RESULT")
        save_status = gr.Textbox(label="Status", lines=2)

    recognize_button.click(fn=recognize_uploaded_photo, inputs=input_photo,
                           outputs=[output_photo, result_text])
    save_button.click(fn=log_result, inputs=[input_photo, actual_people, present_names],
                      outputs=save_status)

# LAB8_LAUNCH_UI=0 lets us execute the whole notebook headless (CI / report generation).
if os.environ.get("LAB8_LAUNCH_UI", "1") == "1":
    # In Colab the app is shown inside the notebook output. Avoid share=True: a public link would
    # expose the biometric database to anyone who gets the URL.
    demo.launch(debug=False)
else:
    print("UI launch skipped (headless run). Set LAB8_LAUNCH_UI=1 or run in Colab to open it.")

---
# ASSIGNMENT 3 — Iris recognition
**Pipeline:** grayscale → CLAHE contrast → **segment** pupil & iris (two circles) →
**normalise** the ring into a rectangle ("rubber sheet") → features → compare.

*Analogy:* the iris is a ring-shaped donut. We cut the donut and unroll it into a strip, so that
a bigger/smaller pupil or a slightly rotated eye still gives the same strip (just shifted).

Two feature methods are compared:
1. **IrisCode (Daugman-style)** — Gabor filters turn the texture into bits; compare with the
   **Hamming distance** (fraction of different bits). Same eye ≈ 0.1–0.3, different eyes ≈ 0.4–0.5.
2. **ORB** (the lab's example) — generic corner-like keypoints + descriptor matching.

In [ ]:
# --- 3.1 Load the dataset and list identities ---
ensure_dataset(IRIS_DIR)
iris_ids = sorted(p.name for p in IRIS_DIR.iterdir() if p.is_dir())
iris_images = {i: sorted(str(p) for p in (IRIS_DIR / i).iterdir() if p.suffix.lower() in IMG_EXT)
               for i in iris_ids}
for i in iris_ids:
    print(i, "—", len(iris_images[i]), "images")

# --- 3.2 Show one sample per identity ---
plt.figure(figsize=(15, 3))
for k, i in enumerate(iris_ids):
    plt.subplot(1, len(iris_ids), k + 1)
    plt.imshow(cv2.imread(iris_images[i][0], cv2.IMREAD_GRAYSCALE), cmap="gray")
    plt.title(i); plt.axis("off")
plt.suptitle("Iris dataset samples (CASIA-IrisV1)")
save_fig("iris_samples", "iris")

In [ ]:
# --- 3.3 Preprocessing: grayscale + CLAHE (Contrast Limited Adaptive Histogram Equalization) ---
# CLAHE boosts local contrast tile-by-tile, so faint iris furrows become visible
# without blowing out the bright sclera.
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def iris_preprocess(path):
    image = cv2.imread(path)
    if image is None:
        raise ValueError(f"Image not found: {path}")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return gray, clahe.apply(gray)

gray, enhanced = iris_preprocess(iris_images[iris_ids[0]][0])
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1); plt.imshow(gray, cmap="gray"); plt.title("Original Image"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(enhanced, cmap="gray"); plt.title("Enhanced Image (CLAHE)"); plt.axis("off")
save_fig("iris_preprocessing", "iris")

In [ ]:
# --- 3.4 Segmentation: find the pupil (inner circle) and the iris (outer circle) ---
def segment_iris(gray):
    """Return pupil (x, y, r) and iris (x, y, r).
    Pupil: the darkest round blob -> threshold + contours + circularity test.
    Iris : Hough circle transform searching radii 1.6-3.8x the pupil radius around the pupil.
    """
    blur = cv2.medianBlur(gray, 7)
    _, dark = cv2.threshold(blur, 60, 255, cv2.THRESH_BINARY_INV)        # pupil pixels -> white
    dark = cv2.morphologyEx(dark, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))  # remove eyelash specks
    contours, _ = cv2.findContours(dark, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    best = None
    for c in contours:
        area = cv2.contourArea(c)
        if area < 300: continue
        (x, y), r = cv2.minEnclosingCircle(c)
        circularity = area / (np.pi * r * r)            # 1.0 for a perfect circle
        if circularity > 0.6 and (best is None or area > best[0]):
            best = (area, x, y, r)
    if best is None:   # fallback: Hough on the whole image
        h, w = gray.shape; best = (0, w / 2, h / 2, min(h, w) / 8)
    _, px, py, pr = best

    circles = cv2.HoughCircles(cv2.GaussianBlur(gray, (9, 9), 2), cv2.HOUGH_GRADIENT, dp=1,
                               minDist=200, param1=40, param2=20,
                               minRadius=int(pr * 1.6), maxRadius=int(min(pr * 3.8, 140)))
    iris = None
    if circles is not None:
        for cx, cy, r in circles[0]:
            if np.hypot(cx - px, cy - py) < pr * 0.5:   # iris must be (almost) concentric with pupil
                iris = (float(cx), float(cy), float(r)); break
    if iris is None:
        iris = (px, py, pr * 2.4)                        # anatomical fallback ratio
    return (px, py, pr), iris

def draw_segmentation(gray, pupil, iris):
    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    cv2.circle(vis, (int(pupil[0]), int(pupil[1])), int(pupil[2]), (255, 0, 0), 2)
    cv2.circle(vis, (int(iris[0]), int(iris[1])), int(iris[2]), (0, 255, 0), 2)
    return vis

def iris_crop(gray, pupil, iris):
    """Segmented iris region: keep the ring between the two circles, black elsewhere, then crop."""
    mask = np.zeros_like(gray)
    cv2.circle(mask, (int(iris[0]), int(iris[1])), int(iris[2]), 255, -1)
    cv2.circle(mask, (int(pupil[0]), int(pupil[1])), int(pupil[2]), 0, -1)
    ring = cv2.bitwise_and(gray, gray, mask=mask)
    x, y, r = int(iris[0]), int(iris[1]), int(iris[2])
    return ring[max(0, y - r): y + r, max(0, x - r): x + r]

# Segment every image, save the crops (the lab's "iris_crops" folder) and show the result
iris_data = {}   # path -> dict with all intermediate results
for i in iris_ids:
    for path in iris_images[i]:
        gray, enhanced = iris_preprocess(path)
        pupil, iris = segment_iris(gray)
        crop = iris_crop(enhanced, pupil, iris)
        crop_path = RESULTS_DIR / "iris" / "crops" / f"{i}_{Path(path).stem}.jpg"
        cv2.imwrite(str(crop_path), crop)
        iris_data[path] = {"id": i, "gray": gray, "enhanced": enhanced, "pupil": pupil,
                           "iris": iris, "crop": crop, "crop_path": str(crop_path)}

paths = list(iris_data)
plt.figure(figsize=(15, 3 * ((len(paths) + 4) // 5)))
for k, p in enumerate(paths):
    d = iris_data[p]
    plt.subplot((len(paths) + 4) // 5, 5, k + 1)
    plt.imshow(draw_segmentation(d["gray"], d["pupil"], d["iris"]))
    plt.title(f"{d['id']}/{Path(p).name}", fontsize=9); plt.axis("off")
plt.suptitle("Segmentation: red = pupil boundary, green = iris boundary")
save_fig("iris_segmentation", "iris")

In [ ]:
# --- 3.5 Normalisation (Daugman rubber-sheet model) + IrisCode features ---
NORM_H, NORM_W = 64, 512    # 64 radial samples x 512 angular samples

def normalize_iris(gray, pupil, iris):
    """Unroll the ring into a NORM_H x NORM_W rectangle.
    Row 0 = pupil edge, last row = iris edge; column = angle 0..360 degrees."""
    theta = np.linspace(0, 2 * np.pi, NORM_W, endpoint=False)
    r = np.linspace(0, 1, NORM_H)[:, None]
    xp, yp = pupil[0] + pupil[2] * np.cos(theta), pupil[1] + pupil[2] * np.sin(theta)
    xi, yi = iris[0] + iris[2] * np.cos(theta), iris[1] + iris[2] * np.sin(theta)
    X = ((1 - r) * xp + r * xi).astype(np.float32)   # linear interpolation between the circles
    Y = ((1 - r) * yp + r * yi).astype(np.float32)
    return cv2.remap(gray, X, Y, cv2.INTER_LINEAR)

def iris_code(norm):
    """2-bit-per-pixel phase code from Gabor filters at two wavelengths, plus a noise mask."""
    n = norm.astype(np.float32)
    n = (n - n.mean()) / (n.std() + 1e-6)
    bits = []
    for wavelength in (16, 32):
        even = cv2.getGaborKernel((31, 31), 4, 0, wavelength, 0.5, 0, ktype=cv2.CV_32F)
        odd = cv2.getGaborKernel((31, 31), 4, 0, wavelength, 0.5, np.pi / 2, ktype=cv2.CV_32F)
        bits += [cv2.filter2D(n, -1, even) > 0, cv2.filter2D(n, -1, odd) > 0]   # sign of real/imag part
    code = np.stack(bits)
    # Mask: ignore the top sector (upper eyelid + eyelashes) and the outer quarter of the ring
    mask = np.ones(norm.shape, bool)
    theta = np.linspace(0, 2 * np.pi, NORM_W, endpoint=False)
    mask[:, (theta > 1.2 * np.pi) & (theta < 1.8 * np.pi)] = False
    mask[NORM_H * 3 // 4:, :] = False
    return code, mask

def hamming_distance(code1, mask1, code2, mask2, max_shift=16):
    """Fraction of disagreeing bits; the strip is circularly shifted to undo head/eye rotation."""
    best = 1.0
    for s in range(-max_shift, max_shift + 1, 2):
        c2, m2 = np.roll(code2, s, axis=2), np.roll(mask2, s, axis=1)
        valid = mask1 & m2
        best = min(best, float((code1 != c2)[:, valid].mean()))
    return best

for p, d in iris_data.items():
    d["norm"] = clahe.apply(normalize_iris(d["gray"], d["pupil"], d["iris"]))
    d["code"], d["mask"] = iris_code(d["norm"])

d = iris_data[paths[0]]
fig, ax = plt.subplots(4, 1, figsize=(12, 7))
ax[0].imshow(d["crop"], cmap="gray"); ax[0].set_title("Segmented iris crop")
ax[1].imshow(d["norm"], cmap="gray"); ax[1].set_title("Normalised strip (rubber sheet, 64 x 512)")
ax[2].imshow(d["code"][0], cmap="gray"); ax[2].set_title("IrisCode bits (one of 4 channels)")
ax[3].imshow(d["mask"], cmap="gray"); ax[3].set_title("Mask (white = usable, black = eyelid / eyelash zone)")
for a in ax: a.axis("off")
plt.tight_layout()
save_fig("iris_normalization_code", "iris")

In [ ]:
# --- 3.6 ORB comparison of segmented crops (the lab's compare_iris, applied to iris_crops) ---
def compare_iris_orb(path1, path2, threshold=IRIS_ORB_THRESHOLD):
    img1 = cv2.imread(path1, cv2.IMREAD_GRAYSCALE); img2 = cv2.imread(path2, cv2.IMREAD_GRAYSCALE)
    if img1 is None or img2 is None:
        raise ValueError("Error loading images")
    orb = cv2.ORB_create(nfeatures=1000)
    kp1, des1 = orb.detectAndCompute(img1, None); kp2, des2 = orb.detectAndCompute(img2, None)
    if des1 is None or des2 is None:
        return 0, "NO MATCH"
    pairs = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False).knnMatch(des1, des2, k=2)
    # Lowe's ratio test: keep a match only if it is clearly better than the 2nd best candidate
    good = [m for pair in pairs if len(pair) == 2 for m, n in [pair] if m.distance < 0.75 * n.distance]
    score = len(good)
    return score, "MATCH" if score >= threshold else "NO MATCH"

def compare_iris(path1, path2):
    """Main iris verifier: IrisCode Hamming distance (lower = more similar)."""
    a, b = iris_data[path1], iris_data[path2]
    hd = hamming_distance(a["code"], a["mask"], b["code"], b["mask"])
    return hd, "MATCH" if hd < IRIS_HD_THRESHOLD else "NO MATCH"

### 3.7 Mandatory experiment — 5 genuine + 5 impostor pairs → `iris_results.csv`
Genuine pairs use **different sessions** (image_01 vs image_03 — taken weeks apart), which is the
realistic and harder case. Impostor pairs compare image_01 of neighbouring identities.

In [ ]:
def mandatory_pairs(images_by_id, genuine_idx=(0, -1)):
    ids = list(images_by_id)
    gen = [(images_by_id[i][genuine_idx[0]], images_by_id[i][genuine_idx[1]], "SAME") for i in ids[:5]]
    imp = [(images_by_id[ids[k]][0], images_by_id[ids[(k + 1) % len(ids)]][0], "DIFFERENT")
           for k in range(min(5, len(ids)))]
    return gen + imp

iris_rows = []
for n, (p1, p2, truth) in enumerate(mandatory_pairs(iris_images), start=1):
    hd, decision = compare_iris(p1, p2)
    orb_score, orb_decision = compare_iris_orb(iris_data[p1]["crop_path"], iris_data[p2]["crop_path"])
    iris_rows.append({
        "pair": n, "image_1": str(Path(p1).relative_to(DATA_DIR)), "image_2": str(Path(p2).relative_to(DATA_DIR)),
        "ground_truth": truth, "hamming_distance": round(hd, 4), "decision": decision,
        "correct": (decision == "MATCH") == (truth == "SAME"),
        "orb_score": orb_score, "orb_decision": orb_decision,
        "orb_correct": (orb_decision == "MATCH") == (truth == "SAME")})
iris_results = pd.DataFrame(iris_rows)
iris_results.to_csv(PROJECT_ROOT / "iris_results.csv", index=False)
print(f"IrisCode accuracy: {iris_results['correct'].mean() * 100:.0f}%   "
      f"ORB accuracy: {iris_results['orb_correct'].mean() * 100:.0f}%")
iris_results

In [ ]:
# --- 3.8 Extended evaluation: ALL possible pairs -> score distributions, FAR / FRR ---
all_rows = []
for p1, p2 in itertools.combinations(paths, 2):
    truth = "SAME" if iris_data[p1]["id"] == iris_data[p2]["id"] else "DIFFERENT"
    hd, _ = compare_iris(p1, p2)
    orb_score, _ = compare_iris_orb(iris_data[p1]["crop_path"], iris_data[p2]["crop_path"])
    all_rows.append({"image_1": Path(p1).parent.name + "/" + Path(p1).name,
                     "image_2": Path(p2).parent.name + "/" + Path(p2).name,
                     "ground_truth": truth, "hamming_distance": hd, "orb_score": orb_score})
iris_all = pd.DataFrame(all_rows)
iris_all.to_csv(RESULTS_DIR / "iris" / "iris_all_pairs.csv", index=False)

def rates(df, score, threshold, higher_is_match):
    match = df[score] >= threshold if higher_is_match else df[score] < threshold
    same = df["ground_truth"] == "SAME"
    far = (match & ~same).sum() / max((~same).sum(), 1) * 100   # strangers accepted
    frr = (~match & same).sum() / max(same.sum(), 1) * 100      # owners rejected
    acc = (match == same).mean() * 100
    return far, frr, acc

IRIS_FAR, IRIS_FRR, IRIS_ACC = rates(iris_all, "hamming_distance", IRIS_HD_THRESHOLD, False)
IRIS_ORB_FAR, IRIS_ORB_FRR, IRIS_ORB_ACC = rates(iris_all, "orb_score", IRIS_ORB_THRESHOLD, True)
gen_hd = iris_all[iris_all.ground_truth == "SAME"].hamming_distance
imp_hd = iris_all[iris_all.ground_truth == "DIFFERENT"].hamming_distance
print(f"Genuine HD: {gen_hd.min():.3f}-{gen_hd.max():.3f} | impostor HD: {imp_hd.min():.3f}-{imp_hd.max():.3f}")
print(f"Threshold {IRIS_HD_THRESHOLD} is placed just below the closest impostor -> security-first (FAR≈0)")
print(f"All {len(iris_all)} pairs ({(iris_all.ground_truth == 'SAME').sum()} genuine):")
print(f"  IrisCode  HD<{IRIS_HD_THRESHOLD}:  accuracy {IRIS_ACC:.1f}%  FAR {IRIS_FAR:.1f}%  FRR {IRIS_FRR:.1f}%")
print(f"  ORB  score>={IRIS_ORB_THRESHOLD}:   accuracy {IRIS_ORB_ACC:.1f}%  FAR {IRIS_ORB_FAR:.1f}%  FRR {IRIS_ORB_FRR:.1f}%")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for truth, color in [("SAME", "tab:green"), ("DIFFERENT", "tab:red")]:
    sub = iris_all[iris_all.ground_truth == truth]
    ax[0].hist(sub.hamming_distance, bins=15, alpha=.7, color=color, label=truth)
    ax[1].hist(sub.orb_score, bins=15, alpha=.7, color=color, label=truth)
ax[0].axvline(IRIS_HD_THRESHOLD, c="k", ls="--"); ax[0].set_title("IrisCode Hamming distance (lower = same)")
ax[1].axvline(IRIS_ORB_THRESHOLD, c="k", ls="--"); ax[1].set_title("ORB good matches (higher = same)")
for a in ax: a.legend()
save_fig("iris_score_distributions", "iris")

In [ ]:
# --- 3.9 Error analysis: show the worst genuine and the closest impostor pair ---
worst_gen = iris_all[iris_all.ground_truth == "SAME"].sort_values("hamming_distance").iloc[-1]
best_imp = iris_all[iris_all.ground_truth == "DIFFERENT"].sort_values("hamming_distance").iloc[0]
plt.figure(figsize=(12, 5))
for k, (row, label) in enumerate([(worst_gen, "hardest genuine pair"), (best_imp, "closest impostor pair")]):
    for j, col in enumerate(["image_1", "image_2"]):
        p = str(IRIS_DIR / row[col])
        dd = iris_data[p]
        plt.subplot(2, 2, 2 * k + j + 1)
        plt.imshow(draw_segmentation(dd["gray"], dd["pupil"], dd["iris"]))
        plt.title(f"{label}: {row[col]}  HD={row.hamming_distance:.3f}", fontsize=9); plt.axis("off")
plt.tight_layout()
save_fig("iris_error_analysis", "iris")
print("Typical reasons for errors: eyelids/eyelashes covering the iris, pupil dilation between "
      "sessions, imperfect circle fits (iris boundary is low-contrast), different focus/illumination.")

---
# ASSIGNMENT 4 — Fingerprint recognition
**Pipeline:** grayscale → Gaussian blur (noise) → CLAHE (contrast) → Otsu binarisation →
skeleton → **minutiae** (ridge endings & bifurcations) for analysis, and **ORB + RANSAC** for matching.

*Analogy:* ridges are like roads on a map. **Minutiae** are the places where a road ends or
splits into two. Two prints match when many such landmarks line up after rotating/shifting one map.

In [ ]:
# --- 4.1 Load the dataset ---
ensure_dataset(FINGER_DIR)
finger_ids = sorted(p.name for p in FINGER_DIR.iterdir() if p.is_dir())
finger_images = {f: sorted(str(p) for p in (FINGER_DIR / f).iterdir() if p.suffix.lower() in IMG_EXT)
                 for f in finger_ids}
for f in finger_ids:
    print(f, "—", len(finger_images[f]), "images")

plt.figure(figsize=(15, 6))
for k, f in enumerate(finger_ids):
    for j, p in enumerate(finger_images[f][:2]):
        plt.subplot(2, len(finger_ids), j * len(finger_ids) + k + 1)
        plt.imshow(cv2.imread(p, cv2.IMREAD_GRAYSCALE), cmap="gray")
        plt.title(f"{f}/{Path(p).name}", fontsize=9); plt.axis("off")
plt.suptitle("Fingerprint samples (SOCOFing): top = real print, bottom = variant")
save_fig("fingerprint_samples", "fingerprint")

In [ ]:
# --- 4.2 Preprocessing: grayscale -> Gaussian blur -> CLAHE -> Otsu binary -> skeleton ---
def fp_preprocess(path, upscale=FP_UPSCALE):
    image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Image not found: {path}")
    image = cv2.resize(image, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC)
    filtered = cv2.GaussianBlur(image, (5, 5), 0)                    # remove sensor noise
    enhanced = clahe.apply(filtered)                                  # equalise ridge contrast
    # Otsu picks the threshold automatically; INV -> ridges (dark) become white (foreground)
    _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))   # remove salt noise
    # drop tiny blobs (dust / pores) that would create fake ridge endings
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    keep = np.isin(labels, np.where(stats[:, cv2.CC_STAT_AREA] >= 60)[0][1:])
    binary = np.where(keep, 255, 0).astype(np.uint8)
    skeleton = skeletonize(binary > 0)                                # ridges 1 pixel thick
    # foreground mask = the finger area (minutiae on its border are just where ridges get cut)
    mask = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))
    mask = cv2.erode(mask, np.ones((25, 25), np.uint8)) > 0
    return {"image": image, "filtered": filtered, "enhanced": enhanced, "binary": binary,
            "skeleton": skeleton, "mask": mask}

def extract_minutiae(fp, min_dist=3 * FP_UPSCALE):
    """Crossing-number method: for each ridge pixel count 0->1 transitions among its 8 neighbours.
       CN == 1 -> ridge ending, CN == 3 -> bifurcation.
    Post-processing (otherwise you get hundreds of fake minutiae):
       * only inside the finger mask (not at the cut-off border of the print)
       * remove minutiae closer than `min_dist` px to another one - short spurs, small gaps and
         bridges always create such tight clusters, real minutiae are spread out."""
    sk, mask = fp["skeleton"].astype(np.uint8), fp["mask"]
    h, w = sk.shape
    border = 5 * FP_UPSCALE                       # image edge: ridges are cut there, not real minutiae
    offsets = [(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]
    points, kinds = [], []
    ys, xs = np.nonzero(sk[1:-1, 1:-1])
    for y, x in zip(ys + 1, xs + 1):
        if not mask[y, x] or min(x, y, w - 1 - x, h - 1 - y) < border:
            continue
        ring = [sk[y + dy, x + dx] for dy, dx in offsets]
        cn = sum(abs(int(ring[i]) - int(ring[(i + 1) % 8])) for i in range(8)) // 2
        if cn in (1, 3):
            points.append((x, y)); kinds.append(cn)
    if len(points) > 1:
        pts = np.array(points, float)
        dist = np.sqrt(((pts[:, None] - pts[None]) ** 2).sum(-1))
        np.fill_diagonal(dist, np.inf)
        isolated = dist.min(axis=1) >= min_dist
        points = [p for p, ok in zip(points, isolated) if ok]
        kinds = [k for k, ok in zip(kinds, isolated) if ok]
    endings = [p for p, k in zip(points, kinds) if k == 1]
    bifurcations = [p for p, k in zip(points, kinds) if k == 3]
    return endings, bifurcations

fp = fp_preprocess(finger_images[finger_ids[0]][0])
endings, bifurcations = extract_minutiae(fp)
titles = ["Original Fingerprint", "Filtered Image", "CLAHE Enhanced", "Binary Image (Otsu)", "Skeleton + minutiae"]
plt.figure(figsize=(18, 4))
for k, key in enumerate(["image", "filtered", "enhanced", "binary", "skeleton"]):
    plt.subplot(1, 5, k + 1); plt.imshow(fp[key], cmap="gray"); plt.title(titles[k]); plt.axis("off")
if endings:
    plt.scatter(*zip(*endings), s=25, facecolors="none", edgecolors="red", label=f"endings ({len(endings)})")
if bifurcations:
    plt.scatter(*zip(*bifurcations), s=25, facecolors="none", edgecolors="lime", label=f"bifurcations ({len(bifurcations)})")
plt.legend(loc="lower right", fontsize=8)
save_fig("fingerprint_preprocessing", "fingerprint")

In [ ]:
# --- 4.3 Matching: ORB keypoints + ratio test (lab) + RANSAC geometric verification (ours) ---
def compare_fingerprints(path1, path2):
    """Return dict with ORB good-match score, RANSAC inlier score and both decisions.
    Why RANSAC? Random look-alike ridge patches give some 'good' ORB matches even between
    different fingers. RANSAC keeps only matches that agree on ONE common rotation+shift,
    which is what a real re-placement of the same finger produces."""
    img1, img2 = fp_preprocess(path1)["enhanced"], fp_preprocess(path2)["enhanced"]
    orb = cv2.ORB_create(nfeatures=1500)
    kp1, des1 = orb.detectAndCompute(img1, None); kp2, des2 = orb.detectAndCompute(img2, None)
    result = {"orb_score": 0, "inliers": 0, "orb_decision": "NO MATCH", "decision": "NO MATCH",
              "kp1": kp1, "kp2": kp2, "good": [], "inlier_mask": None}
    if des1 is None or des2 is None:
        return result
    matches = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False).knnMatch(des1, des2, k=2)
    good = [m for pair in matches if len(pair) == 2 for m, n in [pair] if m.distance < 0.75 * n.distance]
    result.update(orb_score=len(good), good=good,
                  orb_decision="MATCH" if len(good) >= FP_ORB_THRESHOLD else "NO MATCH")
    if len(good) >= 4:
        src = np.float32([kp1[m.queryIdx].pt for m in good]); dst = np.float32([kp2[m.trainIdx].pt for m in good])
        _, inlier_mask = cv2.estimateAffinePartial2D(src, dst, method=cv2.RANSAC, ransacReprojThreshold=8)
        if inlier_mask is not None:
            result.update(inliers=int(inlier_mask.sum()), inlier_mask=inlier_mask.ravel())
    result["decision"] = "MATCH" if result["inliers"] >= FP_INLIER_THRESHOLD else "NO MATCH"
    return result

def draw_fp_matches(path1, path2, res, title):
    img1, img2 = fp_preprocess(path1)["enhanced"], fp_preprocess(path2)["enhanced"]
    keep = [m for k, m in enumerate(res["good"]) if res["inlier_mask"] is not None and res["inlier_mask"][k]]
    vis = cv2.drawMatches(img1, res["kp1"], img2, res["kp2"], keep[:60], None,
                          matchColor=(0, 255, 0), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.imshow(vis); plt.title(title, fontsize=10); plt.axis("off")

f0, f1 = finger_ids[0], finger_ids[1]
g = compare_fingerprints(finger_images[f0][0], finger_images[f0][1])
i = compare_fingerprints(finger_images[f0][0], finger_images[f1][0])
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1); draw_fp_matches(finger_images[f0][0], finger_images[f0][1], g,
                                      f"SAME finger: ORB={g['orb_score']}, inliers={g['inliers']} -> {g['decision']}")
plt.subplot(1, 2, 2); draw_fp_matches(finger_images[f0][0], finger_images[f1][0], i,
                                      f"DIFFERENT fingers: ORB={i['orb_score']}, inliers={i['inliers']} -> {i['decision']}")
save_fig("fingerprint_matching_example", "fingerprint")

### 4.4 Mandatory experiment — 5 genuine + 5 impostor pairs → `fingerprint_results.csv`

In [ ]:
fp_rows = []
for n, (p1, p2, truth) in enumerate(mandatory_pairs(finger_images, genuine_idx=(0, 1)), start=1):
    r = compare_fingerprints(p1, p2)
    e1, b1 = extract_minutiae(fp_preprocess(p1)); e2, b2 = extract_minutiae(fp_preprocess(p2))
    fp_rows.append({
        "pair": n, "image_1": str(Path(p1).relative_to(DATA_DIR)), "image_2": str(Path(p2).relative_to(DATA_DIR)),
        "ground_truth": truth, "orb_score": r["orb_score"], "orb_decision": r["orb_decision"],
        "ransac_inliers": r["inliers"], "decision": r["decision"],
        "correct": (r["decision"] == "MATCH") == (truth == "SAME"),
        "orb_correct": (r["orb_decision"] == "MATCH") == (truth == "SAME"),
        "minutiae_1": len(e1) + len(b1), "minutiae_2": len(e2) + len(b2)})
fp_results = pd.DataFrame(fp_rows)
fp_results.to_csv(PROJECT_ROOT / "fingerprint_results.csv", index=False)
print(f"ORB+RANSAC accuracy: {fp_results['correct'].mean() * 100:.0f}%   "
      f"ORB-only accuracy: {fp_results['orb_correct'].mean() * 100:.0f}%")
fp_results

In [ ]:
# --- 4.5 Extended evaluation: all pairs ---
fp_paths = [p for f in finger_ids for p in finger_images[f]]
fp_all = []
for p1, p2 in itertools.combinations(fp_paths, 2):
    r = compare_fingerprints(p1, p2)
    fp_all.append({"image_1": f"{Path(p1).parent.name}/{Path(p1).name}", "image_2": f"{Path(p2).parent.name}/{Path(p2).name}",
                   "ground_truth": "SAME" if Path(p1).parent == Path(p2).parent else "DIFFERENT",
                   "orb_score": r["orb_score"], "ransac_inliers": r["inliers"]})
fp_all = pd.DataFrame(fp_all)
fp_all.to_csv(RESULTS_DIR / "fingerprint" / "fingerprint_all_pairs.csv", index=False)
FP_FAR, FP_FRR, FP_ACC = rates(fp_all, "ransac_inliers", FP_INLIER_THRESHOLD, True)
FP_ORB_FAR, FP_ORB_FRR, FP_ORB_ACC = rates(fp_all, "orb_score", FP_ORB_THRESHOLD, True)
imp = fp_all[fp_all.ground_truth == "DIFFERENT"]
print(f"All {len(fp_all)} pairs: ORB+RANSAC acc {FP_ACC:.1f}% (FAR {FP_FAR:.1f}%, FRR {FP_FRR:.1f}%) | "
      f"ORB-only acc {FP_ORB_ACC:.1f}% (FAR {FP_ORB_FAR:.1f}%, FRR {FP_ORB_FRR:.1f}%)")
print(f"Highest impostor scores: ORB={imp.orb_score.max()}, inliers={imp.ransac_inliers.max()} "
      f"-> the lab's ORB threshold {FP_ORB_THRESHOLD} has only a small safety margin")

### 4.6 Stress test — simulated re-capture
SOCOFing's variants are synthetic edits of the **same** scan, so genuine pairs are easier than in
real life (a real second touch has different position, pressure, rotation and partial area).
We simulate this by rotating, shifting, cropping and adding noise, and watch the score drop.

In [ ]:
def simulate_recapture(path, angle, crop, noise, out_path, smudge=0):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if smudge:     # wet / moving finger -> ridges smear together (Gaussian blur)
        img = cv2.GaussianBlur(img, (0, 0), smudge)
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    M[:, 2] += np.random.uniform(-4, 4, 2)                               # small shift
    img = cv2.warpAffine(img, M, (w, h), borderValue=255)
    ch, cw = int(h * crop), int(w * crop)                                 # keep only part of the finger
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    canvas = np.full_like(img, 255); canvas[y0:y0 + ch, x0:x0 + cw] = img[y0:y0 + ch, x0:x0 + cw]
    canvas = np.clip(canvas + np.random.normal(0, noise, canvas.shape), 0, 255).astype(np.uint8)
    cv2.imwrite(out_path, canvas)
    return out_path

stress_dir = RESULTS_DIR / "fingerprint" / "stress"
stress_dir.mkdir(exist_ok=True)
# (name, rotation°, fraction of the print kept, noise std, smudge sigma)
levels = [("mild", 5, 0.95, 5, 0), ("medium", 15, 0.8, 12, 0), ("hard", 30, 0.65, 20, 0.8),
          ("extreme", 45, 0.5, 35, 1.2)]
stress_rows = []
for f in finger_ids:
    original = finger_images[f][0]
    for name, angle, crop, noise, smudge in levels:
        sim = simulate_recapture(original, angle, crop, noise, str(stress_dir / f"{f}_{name}.bmp"), smudge)
        r = compare_fingerprints(original, sim)
        stress_rows.append({"finger": f, "level": name, "rotation_deg": angle, "area_kept": crop,
                            "noise_std": noise, "smudge_sigma": smudge, "orb_score": r["orb_score"], "ransac_inliers": r["inliers"],
                            "decision": r["decision"]})
stress = pd.DataFrame(stress_rows)
stress.to_csv(RESULTS_DIR / "fingerprint" / "stress_test.csv", index=False)
plt.figure(figsize=(15, 3.5))
plt.subplot(1, len(levels) + 1, 1)
plt.imshow(cv2.imread(finger_images[finger_ids[0]][0], cv2.IMREAD_GRAYSCALE), cmap="gray")
plt.title("original"); plt.axis("off")
for k, (name, *_ ) in enumerate(levels):
    row = stress[(stress.finger == finger_ids[0]) & (stress.level == name)].iloc[0]
    plt.subplot(1, len(levels) + 1, k + 2)
    plt.imshow(cv2.imread(str(stress_dir / f"{finger_ids[0]}_{name}.bmp"), cv2.IMREAD_GRAYSCALE), cmap="gray")
    plt.title(f"{name}: inliers={row.ransac_inliers}\n{row.decision}", fontsize=9); plt.axis("off")
save_fig("fingerprint_stress_examples", "fingerprint")
print(stress.groupby("level", sort=False)[["orb_score", "ransac_inliers"]].mean().round(1))
print("Genuine acceptance by level:\n", stress.groupby("level", sort=False)["decision"]
      .apply(lambda s: f"{(s == 'MATCH').mean() * 100:.0f}%"))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for truth, color in [("SAME", "tab:green"), ("DIFFERENT", "tab:red")]:
    ax[0].hist(fp_all[fp_all.ground_truth == truth].ransac_inliers, bins=20, alpha=.7, color=color, label=truth)
ax[0].axvline(FP_INLIER_THRESHOLD, c="k", ls="--"); ax[0].set_xscale("symlog"); ax[0].legend()
ax[0].set_title("RANSAC inliers, all dataset pairs (log scale)")
stress.boxplot(column="ransac_inliers", by="level", ax=ax[1])
ax[1].axhline(FP_INLIER_THRESHOLD, c="k", ls="--"); ax[1].set_title("Stress test: genuine score vs distortion")
plt.suptitle("")
save_fig("fingerprint_scores_and_stress", "fingerprint")

---
# 5. Comparative analysis of the three biometric methods
Built from the numbers computed above, so it updates automatically when the data changes.

In [ ]:
face_acc_text = f"{FACE_ACCURACY:.1f}% on group photos" if FACE_ACCURACY is not None else "see face_results.csv"
comparison = pd.DataFrame({
    "Criterion": ["Data source", "Algorithm used", "Feature extraction method", "Matching method",
                  "Experimental results", "Main recognition errors", "Method limitations"],
    "Face Recognition": [
        "Custom team dataset (4 registration + 1 test photo per student)",
        f"OpenCV Haar detector + {MODEL_NAME} (deep CNN)",
        "512-D face embedding from aligned face crop",
        f"Cosine distance, nearest neighbour, UNKNOWN if > {FACE_THRESHOLD}",
        f"{face_acc_text}; test_05 identification {test_df['correct'].mean() * 100:.0f}%",
        "Missed small/rotated/dark faces (detector), strangers accepted if threshold too high",
        "Lighting, pose, occlusion; spoofing with photos; privacy of stored templates"],
    "Iris Recognition": [
        "CASIA-IrisV1 subset (5 identities x 3 images)",
        "Threshold+contours (pupil), Hough circle (iris), Daugman rubber sheet",
        "Gabor phase IrisCode (4 x 64 x 512 bits) + mask; ORB on crops for comparison",
        f"Hamming distance with rotation shifts, MATCH if < {IRIS_HD_THRESHOLD}",
        f"All pairs: IrisCode acc {IRIS_ACC:.1f}% (FAR {IRIS_FAR:.1f}%, FRR {IRIS_FRR:.1f}%); ORB acc {IRIS_ORB_ACC:.1f}%",
        "Cross-session pairs near threshold; eyelid occlusion; inaccurate outer circle",
        "Needs NIR camera & cooperative user; segmentation is the weak link; ORB unsuitable"],
    "Fingerprint Recognition": [
        "SOCOFing subset (5 fingers x real + 2 altered variants)",
        "Gaussian + CLAHE + Otsu, skeleton, crossing-number minutiae; ORB + RANSAC",
        "ORB keypoints/descriptors on enhanced ridges (minutiae for analysis)",
        f"Ratio-test matches, RANSAC similarity transform, MATCH if inliers >= {FP_INLIER_THRESHOLD}",
        f"All pairs: ORB+RANSAC acc {FP_ACC:.1f}% (FAR {FP_FAR:.1f}%); ORB-only acc {FP_ORB_ACC:.1f}%",
        "Heavy rotation/partial prints lower scores (stress test); ORB-only impostor scores near 20",
        "Variants are synthetic -> optimistic; low resolution (96x103); no real minutiae matcher"],
})
comparison.to_csv(RESULTS_DIR / "comparison" / "comparative_analysis.csv", index=False)
try:
    md_table = comparison.to_markdown(index=False)      # needs the 'tabulate' package
except ImportError:
    md_table = comparison.to_string(index=False)
(RESULTS_DIR / "comparison" / "comparative_analysis.md").write_text(md_table)
comparison

In [ ]:
# Bar chart of the headline accuracies
labels, values = [], []
if FACE_ACCURACY is not None:
    labels.append("Face\n(group photos)"); values.append(FACE_ACCURACY)
labels += ["Face\n(test_05)", "Iris\nIrisCode", "Iris\nORB", "Fingerprint\nORB+RANSAC", "Fingerprint\nORB only"]
values += [test_df["correct"].mean() * 100, IRIS_ACC, IRIS_ORB_ACC, FP_ACC, FP_ORB_ACC]
plt.figure(figsize=(11, 4))
bars = plt.bar(labels, values, color=["#4C72B0"] * (len(values) - 4) + ["#55A868", "#C44E52", "#8172B2", "#CCB974"])
for b, v in zip(bars, values):
    plt.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.0f}%", ha="center")
plt.ylim(0, 110); plt.ylabel("accuracy %"); plt.title("Recognition accuracy by modality / method")
save_fig("accuracy_comparison", "comparison")

---
# 6. Security & privacy notes (AI in Cybersecurity)
* **Biometrics cannot be changed** like a password — a leaked template is leaked forever.
  This notebook keeps templates **only in memory**; nothing is pickled to disk.
* `.gitignore` excludes `data/student_faces/*` and `data/group_photos/*` — real student photos must
  never reach a public repository (lab requirement §8).
* Don't use `share=True` in Gradio for the demo: it creates a public URL to the system.
* Threats: presentation attacks (printed photo / screen replay / fake finger), template theft,
  database poisoning during registration, adversarial images. Mitigations: liveness detection,
  template encryption / cancellable biometrics, access control and audit logs, multi-factor or
  multi-modal fusion (e.g. face + fingerprint).